# Agentic DocQA — QASPER, 50 questions, Rudra

Fresh test-set sample, seed 42, per-paper mode. Answer, formatter and judge: `qwen2.5:7b`. The judge is the same model as the answerer. Sample: 29 extractive, 9 abstractive, 7 boolean, 5 unanswerable. Production agent prompts and routing remain unchanged.

Use Rudra Paliwal (`paliwalrudra2025@gmail.com`). Results go into the shared Agentic DocQA Drive folder through the Drive API. Each completed question is uploaded, downloaded and checksum-verified before the next starts. The first question runs separately so its cloud backup can be checked before continuing.


## GPU
Select Runtime → Change runtime type → T4 GPU.


In [ ]:
import subprocess, sys, os, json, time, hashlib, io, shutil, stat, zipfile, urllib.request
from pathlib import Path, PurePosixPath
check=subprocess.run(["nvidia-smi"],text=True,capture_output=True)
print(check.stdout or check.stderr)
if check.returncode:
    raise RuntimeError("Choose a GPU runtime before evaluation.")
EXPECTED_ACCOUNT="paliwalrudra2025@gmail.com"
SHARED_FOLDER="1ialgDUXlLzdGv_4_47aitvlNGZzw9NRV"
BACKUP_PARENT="1Qrs6GoN-Pa_PuJLFO3NW1ycAw--legXv"
SOURCE_NAME="agentic-docqa-qasper-50-20261102.zip"
SOURCE_SHA256=""


## Authorize Rudra and load the source
Google may ask you to authorize Colab. Select Rudra. The account check prevents saving into a different profile. Credentials are not printed or saved with results.


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
auth.authenticate_user()
credentials,_=google.auth.default()
drive_api=build("drive","v3",credentials=credentials,cache_discovery=False)
actual=drive_api.about().get(fields="user(emailAddress)").execute()["user"]["emailAddress"]
assert actual.casefold()==EXPECTED_ACCOUNT.casefold(), f"Select {EXPECTED_ACCOUNT}; authenticated as {actual}"
files=drive_api.files().list(q=f"'{SHARED_FOLDER}' in parents and name = '{SOURCE_NAME}' and trashed = false",fields="files(id,name)").execute()["files"]
assert len(files)==1, "Expected one fresh source ZIP in the shared folder"
archive_bytes=drive_api.files().get_media(fileId=files[0]["id"]).execute()
assert SOURCE_SHA256 and hashlib.sha256(archive_bytes).hexdigest()==SOURCE_SHA256, "Source ZIP checksum mismatch"
REPO=Path("/content/agentic-docqa-qasper50")/SOURCE_SHA256[:16]
REPO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    roots={PurePosixPath(e.filename).parts[0] for e in archive.infolist()}
    assert len(roots)==1, "Expected one project root"
    for entry in archive.infolist():
        path=PurePosixPath(entry.filename)
        if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(entry.external_attr>>16):
            raise ValueError("Unsafe ZIP entry")
        parts=path.parts[1:]
        if not parts or ".git" in parts: continue
        target=REPO.joinpath(*parts)
        if target.is_symlink() or any(p.is_symlink() for p in target.parents): raise ValueError("Unsafe target")
        if entry.is_dir(): target.mkdir(parents=True,exist_ok=True)
        else:
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(archive.read(entry))
os.chdir(REPO)
sys.path.insert(0,str(REPO))
print("ACCOUNT VERIFIED:",actual)
print("SOURCE VERIFIED:",SOURCE_SHA256)


## Dependencies and cloud persistence preflight
A small test object must survive a Drive API upload and readback before downloading models. All model caches and the working index use the runtime disk.


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","--disable-pip-version-check","-q","-r","requirements.txt","-r","eval/qasper/requirements.txt","google-api-python-client","google-auth"],check=True)
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-q"],check=True)
from eval.qasper.drive_backup import DriveBackup, encoded
backup=DriveBackup(drive_api,BACKUP_PARENT,EXPECTED_ACCOUNT)
preflight_folder=backup.folder(BACKUP_PARENT,"qasper50-rudra-20261102")
receipt=backup.upload(preflight_folder,"cloud-preflight.json",encoded({"account":actual,"source_sha256":SOURCE_SHA256,"questions":50,"seed":42,"verified_via":"Drive API readback"}))
BACKUP_PARENT=preflight_folder
OUT=Path("/content/qasper50-results")
OUT.mkdir(parents=True,exist_ok=True)
CACHE=Path("/content/qasper50-cache")
os.environ.update(OLLAMA_MODELS=str(CACHE/"ollama"),HF_HOME=str(CACHE/"huggingface"),FASTEMBED_CACHE_PATH=str(CACHE/"fastembed"),SENTENCE_TRANSFORMERS_HOME=str(CACHE/"sentence-transformers"))
print("PREFLIGHT CLOUD VERIFIED:",receipt["sha256"])
print("RESULT PARENT: https://drive.google.com/drive/folders/"+BACKUP_PARENT)


## Start Ollama
Install from Ollama’s official source and use one loaded model at a time.


In [ ]:
OLLAMA_URL="http://127.0.0.1:11434"
def ollama_json(path,payload=None,timeout=10):
    request=urllib.request.Request(OLLAMA_URL+path,data=None if payload is None else json.dumps(payload).encode(),headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(request,timeout=timeout) as response: return json.load(response)
def ready():
    try: ollama_json("/api/tags",timeout=2); return True
    except Exception: return False
if not shutil.which("ollama"):
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","zstd"],check=True)
    installer=Path("/content/install-ollama-qasper50.sh")
    urllib.request.urlretrieve("https://ollama.com/install.sh",installer)
    subprocess.run(["sh",str(installer)],check=True)
os.environ.update(OLLAMA_HOST="127.0.0.1:11434",OLLAMA_CONTEXT_LENGTH="8192",OLLAMA_NUM_PARALLEL="1",OLLAMA_MAX_LOADED_MODELS="1")
if not ready():
    log=(OUT/"ollama.log").open("a")
    ollama_process=subprocess.Popen(["ollama","serve"],stdout=log,stderr=subprocess.STDOUT,env=os.environ.copy(),start_new_session=True)
    for _ in range(60):
        if ready(): break
        time.sleep(1)
    else: raise RuntimeError("Ollama did not start")
print("Ollama ready")


## Models and GPU checks
Use the same Qwen 2.5 7B model for answering, formatting and judging. Reranker files are downloaded to local disk at a pinned revision to avoid mounted-cache symlinks.


In [ ]:
ANSWER_MODEL="qwen2.5:7b"
for model in [ANSWER_MODEL,"nomic-embed-text"]:
    subprocess.run(["ollama","pull",model],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
ollama_json("/api/generate",{"model":ANSWER_MODEL,"prompt":"Reply with OK.","stream":False,"keep_alive":"30m","options":{"num_predict":8,"num_ctx":8192}},timeout=600)
loaded=next(m for m in ollama_json("/api/ps")["models"] if m["name"]==ANSWER_MODEL)
assert loaded.get("size_vram",0)>=.9*loaded["size"], "Model substantially offloaded to CPU"
print("ANSWER/JUDGE GPU VERIFIED:",loaded["name"],loaded["size_vram"])
from huggingface_hub import snapshot_download
from safetensors import safe_open
REVISION="2cfc18c9415c912f9d8155881c133215df768a70"
RERANK_DIR=Path("/content/qasper-reranker")/REVISION
snapshot_download("BAAI/bge-reranker-base",revision=REVISION,local_dir=str(RERANK_DIR),token=False,allow_patterns=["config.json","model.safetensors","sentencepiece.bpe.model","tokenizer.json","tokenizer_config.json","special_tokens_map.json"])
with safe_open(str(RERANK_DIR/"model.safetensors"),framework="pt",device="cpu") as weights:
    print("RERANKER WEIGHTS VERIFIED:",len(list(weights.keys())))
os.environ.update(EVAL_LLM_MODEL=ANSWER_MODEL,LLM_MODEL=ANSWER_MODEL,LLM_TEMPERATURE="0",LLM_BASE_URL=OLLAMA_URL+"/v1",LLM_API_KEY="ollama",RERANK_MODEL=str(RERANK_DIR),RERANK_DEVICE="cuda",NOMIC_PREFIXES="true",RETRIEVAL_MODE="hybrid",RERANKER="cross_encoder",LANGFUSE_ENABLED="false")
from app.core.reranker import get_reranker
reranker=get_reranker(kind="cross_encoder")
print("RERANKER DEVICE:",reranker.model.device)
assert str(reranker.model.device).startswith("cuda")


## Prepare the genuine 50-question sample
A five-per-type floor keeps all types represented. The manifest is part of the run fingerprint.


In [ ]:
COMMAND=[sys.executable,"-u","-m","eval.qasper.run","--split","test","--n","50","--minimum-per-type","5","--seed","42","--mode","per_paper","--answer-model",ANSWER_MODEL,"--judge-model",ANSWER_MODEL,"--format-model",ANSWER_MODEL,"--cache-dir",str(CACHE/"qasper"),"--sample-dir",str(CACHE/"samples"),"--no-prefix-ablation","--report-every","5","--out",str(OUT),"--drive-folder-id",BACKUP_PARENT,"--drive-account",EXPECTED_ACCOUNT]
def run_logged(extra,filename):
    with (OUT/filename).open("a") as output:
        proc=subprocess.Popen(COMMAND+extra,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=os.environ.copy())
        for line in proc.stdout:
            print(line,end="",flush=True); output.write(line); output.flush()
        status=proc.wait()
    print("EXIT STATUS:",status)
    return status
assert run_logged(["--prepare-only"],"prepare.log")==0
manifest=json.loads((OUT/"sample_test_50_42.json").read_text())
assert manifest["sampled_n"]==50
backup.upload(BACKUP_PARENT,"sample_test_50_42.json",encoded(manifest),immutable=True)
print("SAMPLE CLOUD VERIFIED:",manifest["quotas"])


## First question — verify its cloud backup
This uses the actual run folder. The remaining run resumes the same experiment and skips this question if complete.


In [ ]:
first_status=run_logged(["--limit","1","--resume"],"first-question.log")
checkpoint=next(OUT.glob("qasper-*.partial.json"))
first=json.loads(checkpoint.read_text())
assert len(first["results"])==1, "First question was not recorded"
assert list((OUT/"cloud-receipts").glob("*.json")), "First question has no verified cloud receipt"
print("FIRST QUESTION CLOUD VERIFIED")
print("CLOUD RUN: https://drive.google.com/drive/folders/"+first["metadata"]["drive_backup_folder"])


## Continue to 50 questions
After independently checking the first question in Drive, enable the switch and run this cell. An upload/readback failure stops evaluation immediately; completed questions can be restored using the same settings and --resume.


In [ ]:
CLOUD_CHECKPOINT_CHECKED=False #@param {type:"boolean"}
if CLOUD_CHECKPOINT_CHECKED:
    final_status=run_logged(["--resume"],"evaluation.log")
    data=json.loads(checkpoint.read_text())
    print("ATTEMPTED:",len(data["results"]),"/ 50")
    print("COMPLETE:",sum(r["status"]=="complete" for r in data["results"].values()))
    print("ERRORS:",sum(r["status"]=="error" for r in data["results"].values()))
    print("CLOUD RUN: https://drive.google.com/drive/folders/"+data["metadata"]["drive_backup_folder"])
else:
    print("Paused after the first question. Check its Drive backup before continuing.")


## Report and independent download
Full JSON includes answers, evidence, raw judgments, retrieval details and metrics. Cloud snapshots and all per-question records remain in Drive. Download a second copy here.


In [ ]:
from IPython.display import Markdown,display
reports=sorted(OUT.glob("*.md"),key=lambda p:p.stat().st_mtime)
if reports:
    display(Markdown(reports[-1].read_text()))
from google.colab import files
archive_path=Path(shutil.make_archive("/content/qasper50-results-backup","zip",OUT))
files.download(str(archive_path))
